# calculations.ipynb — CBAM Cost Estimation: Default Emission Values

Estimates the total CBAM certificate cost for each exporting country and sector,
based on 2024 EU import volumes and EU Commission default emission values.

**Methodology:**
- Estimated cost (EUR) = import volume (tonnes) x default emission value (tCO2/t) x certificate price (EUR/tCO2)
- Default emission values in `default_2026` already include the applicable markup, as confirmed
  by the source xlsx column header 'including mark-up' (10% for most sectors, 1% for fertilizers).
  No additional markup is applied in this notebook.
- Certificate price: EUR 75.36/tCO2 (first official CBAM price, European Commission, April 2026)
- Trade flow year: 2024

**Route scenarios:**
- `_high_route`: uses the highest default_2026 per (country, cn_code) — upper bound cost estimate
- `_low_route`: uses the lowest default_2026 per (country, cn_code) — lower bound cost estimate
- Where only one route exists per (country, cn_code), high and low values are identical
- `has_route_variation` flags rows where high and low differ

**Join logic:**
- cbam_defaults is the left (anchor) table. All 119 countries with published defaults
  are retained regardless of whether they have matching 2024 trade flow data.
- Countries with no trade flow match receive import_tonnes = 0 and cost = 0.
- The 116 trade flow countries with no CBAM default are correctly excluded.
- Sector labels derived from CN code prefixes via SECTOR_MAP.

**Known limitations:**
- 22 countries with published CBAM defaults had no recorded EU imports in 2024.
- CBAM certificate price is parameterized and should be updated as the market matures.
- The markup in default_2026 is specific to 2026. For future-year analysis,
  use default_2027 (20% markup) or default_2028_onwards (30% markup).

**Output tables written to db/cbam.db:**
- `cbam_cost_by_country_sector` — grain: (country, sector, cn_code)
- `cbam_cost_by_country` — grain: (country), aggregated across all sectors
- `cbam_cost_by_sector` — grain: (sector), aggregated across all countries

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
# Standard libraries for data manipulation, database access, and path handling.

import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
# ── Display formatting ────────────────────────────────────────────────────────
# Suppress scientific notation globally for this notebook session.

pd.set_option('display.float_format', '{:,.2f}'.format)

In [3]:
# ── Constants ─────────────────────────────────────────────────────────────────
# All key assumptions defined here as named constants.
#
# CERTIFICATE_PRICE_EUR:
#   First official CBAM certificate price published by the European Commission
#   on 7 April 2026. Tracks the EU ETS and will fluctuate over time.
#   Source: https://www.homaio.com/post/eu-ets-definitions-updated-guide-for-2025
#
# REFERENCE_YEAR:
#   2024 is the most recent complete year and the last full year before
#   CBAM's definitive phase began (1 January 2026).
#
# SECTOR_MAP:
#   Maps CN code prefixes to sector labels. Derived from the defaults table
#   so all 119 countries carry a sector label including zero-trade rows.
#   Longest prefix matched first to avoid shorter keys capturing longer ones.

CERTIFICATE_PRICE_EUR = 75.36   # EUR/tCO2, first official EC CBAM price, April 2026
REFERENCE_YEAR        = 2024    # Trade flow reference year

SECTOR_MAP = {
    '2507': 'Cement',
    '2523': 'Cement',
    '2601': 'Iron and Steel',
    '2716': 'Electricity',
    '2804': 'Hydrogen',
    '2808': 'Fertilizers',
    '2814': 'Fertilizers',
    '2834': 'Fertilizers',
    '3102': 'Fertilizers',
    '3105': 'Fertilizers',
    '6810': 'Cement',
    '6811': 'Cement',
    '72'  : 'Iron and Steel',
    '73'  : 'Iron and Steel',
    '76'  : 'Aluminium',
}

In [4]:
# ── Database connection ───────────────────────────────────────────────────────
# Connect to the SQLite database generated in notebook 08.
# projects/01_country_exposure/ is two levels below the repo root.

DB_PATH = Path('../../db/cbam.db')

assert DB_PATH.exists(), (
    f'Database not found at {DB_PATH.resolve()}.\n'
    f'Run notebook 08 first to generate cbam.db.'
)

con = sqlite3.connect(DB_PATH)
print(f'Connected to: {DB_PATH.resolve()}')

Connected to: /Users/milcahmaryjoseph/Documents/GitHub/cbam-analysis/db/cbam.db


In [5]:
# ── Load CBAM default emission values ────────────────────────────────────────
# Pull all routes per (country, cn_code) — deduplication happens later.
# Rows where default_2026 IS NULL are excluded. One known case: Chile CN 73061100,
# confirmed as a dash in the legally binding regulation (EUR-Lex page 471/2400).

query_defaults = """
    SELECT
        country,
        cn_code,
        production_route_code,
        production_route,
        direct_emissions,
        indirect_emissions,
        total_emissions,
        default_2026
    FROM cbam_defaults
    WHERE default_2026 IS NOT NULL
"""

df_defaults = pd.read_sql(query_defaults, con)
print(f'CBAM default rows loaded: {len(df_defaults):,}')
print(f'Countries with defaults:  {df_defaults["country"].nunique()}')
print(f'Unique CN codes:          {df_defaults["cn_code"].nunique()}')
df_defaults.head()

CBAM default rows loaded: 10,670
Countries with defaults:  119
Unique CN codes:          262


,country,cn_code,production_route_code,production_route,direct_emissions,indirect_emissions,total_emissions,default_2026
0,Albania,25231000,(A),Grey Clinker / Cement,0.87,0.00,0.87,0.96
1,Albania,25232900,NaN,NaN,0.90,0.03,0.93,1.02
2,Albania,25239000,(A),Grey Clinker / Cement,0.86,0.03,0.89,0.98
3,Albania,28080000,NaN,NaN,2.73,0.04,2.76,2.79
4,Albania,28142000,NaN,NaN,0.65,0.03,0.68,0.69


In [6]:
# ── Assign sector labels from CN codes ───────────────────────────────────────
# Sector labels derived here so all 119 countries carry a sector label,
# including those with no trade flow data.
# Longest prefix matched first to avoid ambiguous matches.

def map_sector(cn_code: str) -> str:
    """Return the CBAM sector name for a given CN code string."""
    for prefix in sorted(SECTOR_MAP.keys(), key=len, reverse=True):
        if cn_code.startswith(prefix):
            return SECTOR_MAP[prefix]
    return None

df_defaults['sector'] = df_defaults['cn_code'].apply(map_sector)

unmapped = df_defaults[df_defaults['sector'].isna()]['cn_code'].unique()
if len(unmapped) > 0:
    raise ValueError(
        f'CN codes not matched by SECTOR_MAP: {unmapped}\n'
        f'Add the relevant prefix to SECTOR_MAP in the constants cell.'
    )

print('Sector assignment complete:')
print(df_defaults['sector'].value_counts())

Sector assignment complete:
sector
Iron and Steel    6251
Fertilizers       2389
Aluminium         1608
Cement             329
Hydrogen            93
Name: count, dtype: int64


In [7]:
# ── Build high-route and low-route defaults per (country, cn_code) ────────────
# For each (country, cn_code) combination, extract:
#   - high_route: the row with the highest default_2026 (upper bound cost)
#   - low_route:  the row with the lowest default_2026 (lower bound cost)
#
# Where only one route exists, high and low will be identical.
# production_route_code and production_route are retained for both scenarios
# so the dashboard can display which route drives each bound.

df_high = (
    df_defaults
    .sort_values('default_2026', ascending=False)
    .drop_duplicates(subset=['country', 'cn_code'], keep='first')
    .reset_index(drop=True)
    .rename(columns={
        'default_2026'          : 'default_2026_high_route',
        'production_route_code' : 'production_route_code_high',
        'production_route'      : 'production_route_high',
    })
)

df_low = (
    df_defaults
    .sort_values('default_2026', ascending=True)
    .drop_duplicates(subset=['country', 'cn_code'], keep='first')
    .reset_index(drop=True)
    [['country', 'cn_code', 'default_2026', 'production_route_code', 'production_route']]
    .rename(columns={
        'default_2026'          : 'default_2026_low_route',
        'production_route_code' : 'production_route_code_low',
        'production_route'      : 'production_route_low',
    })
)

# Merge high and low into a single defaults table
df_defaults_both = df_high.merge(
    df_low[['country', 'cn_code',
            'default_2026_low_route',
            'production_route_code_low',
            'production_route_low']],
    on=['country', 'cn_code'],
    how='left'
)

# Flag rows where a cheaper route exists
df_defaults_both['has_route_variation'] = (
    df_defaults_both['default_2026_high_route'] != df_defaults_both['default_2026_low_route']
)

n_varied = df_defaults_both['has_route_variation'].sum()
print(f'Rows before deduplication:        {len(df_defaults):,}')
print(f'Rows after deduplication:         {len(df_defaults_both):,}')
print(f'Rows with route variation:        {n_varied:,}')
print(f'Rows without route variation:     {len(df_defaults_both) - n_varied:,}')

Rows before deduplication:        10,670
Rows after deduplication:         10,641
Rows with route variation:        28
Rows without route variation:     10,613


In [8]:
# ── Load 2024 EU import trade flows (tonnes and value) ────────────────────────
# Pull both QUANTITY_IN_TONNES and VALUE_IN_EUROS for 2024.
# Tonnes are needed for the cost calculation.
# EUR value is needed for cost-as-share-of-export-value metric.
# Both are pivoted into columns so each (country, cn_code) has one row.

query_trade = f"""
    SELECT
        country,
        iso2,
        cn_code,
        indicator,
        value
    FROM trade_flows
    WHERE year = {REFERENCE_YEAR}
      AND indicator IN ('QUANTITY_IN_TONNES', 'VALUE_IN_EUROS')
"""

df_trade_long = pd.read_sql(query_trade, con)

# Pivot indicators into columns
df_trade = (
    df_trade_long
    .pivot_table(
        index   = ['country', 'iso2', 'cn_code'],
        columns = 'indicator',
        values  = 'value',
        aggfunc = 'sum'
    )
    .reset_index()
    .rename(columns={
        'QUANTITY_IN_TONNES': 'import_tonnes',
        'VALUE_IN_EUROS'    : 'import_value_eur'
    })
)
df_trade.columns.name = None

print(f'Trade flow rows after pivot: {len(df_trade):,}')
print(f'Partner countries:           {df_trade["country"].nunique()}')
df_trade.head()

Trade flow rows after pivot: 15,734
Partner countries:           229


,country,iso2,cn_code,import_tonnes,import_value_eur
0,Afghanistan,AF,25070080,0.10,54.00
1,Afghanistan,AF,7210,0.01,433.00
2,Afghanistan,AF,73049000,0.35,"3,411.00"
3,Afghanistan,AF,7308,24.73,"41,197.00"
4,Afghanistan,AF,7309,0.55,"8,665.00"


In [9]:
# ── Join trade flows to CBAM defaults (left join from defaults) ───────────────
# Left join ensures all 119 CBAM countries are retained.
# Countries with no trade flow match receive zeros for tonnes and value.
# iso2 is filled from country_crosswalk for any countries missing it after the join.

df_merged = df_defaults_both.merge(
    df_trade[['country', 'iso2', 'cn_code', 'import_tonnes', 'import_value_eur']],
    on=['country', 'cn_code'],
    how='left'
)

# Fill nulls for unmatched rows
df_merged['import_tonnes']    = df_merged['import_tonnes'].fillna(0)
df_merged['import_value_eur'] = df_merged['import_value_eur'].fillna(0)

# Fill iso2 from crosswalk for countries with no trade flow match
missing_iso2 = df_merged[df_merged['iso2'].isna()]['country'].unique()
if len(missing_iso2) > 0:
    df_crosswalk = pd.read_sql('SELECT country, iso2 FROM country_crosswalk', con)
    df_merged = df_merged.merge(df_crosswalk, on='country', how='left', suffixes=('', '_cw'))
    df_merged['iso2'] = df_merged['iso2'].fillna(df_merged['iso2_cw'])
    df_merged = df_merged.drop(columns='iso2_cw')
    print(f'iso2 filled from crosswalk for {len(missing_iso2)} countries.')

print(f'Rows after join:         {len(df_merged):,}')
print(f'Countries retained:      {df_merged["country"].nunique()}')
print(f'Zero import_tonnes rows: {(df_merged["import_tonnes"] == 0).sum():,}')

iso2 filled from crosswalk for 117 countries.
Rows after join:         10,641
Countries retained:      119
Zero import_tonnes rows: 5,754


In [10]:
# ── Core CBAM cost calculation — both route scenarios ─────────────────────────
# Calculate embedded CO2 and CBAM cost for both high-route and low-route scenarios.
# Formula: import_tonnes x default_2026_[scenario] x CERTIFICATE_PRICE_EUR
# Rows with zero import volume produce zero cost in both scenarios.

df_merged['embedded_co2_high_route'] = (
    df_merged['import_tonnes'] * df_merged['default_2026_high_route']
)
df_merged['cbam_cost_eur_high_route'] = (
    df_merged['embedded_co2_high_route'] * CERTIFICATE_PRICE_EUR
)

df_merged['embedded_co2_low_route'] = (
    df_merged['import_tonnes'] * df_merged['default_2026_low_route']
)
df_merged['cbam_cost_eur_low_route'] = (
    df_merged['embedded_co2_low_route'] * CERTIFICATE_PRICE_EUR
)

print('Total estimated CBAM cost (all countries, all sectors):')
print(f'  High route: EUR {df_merged["cbam_cost_eur_high_route"].sum():>18,.0f}')
print(f'  Low route:  EUR {df_merged["cbam_cost_eur_low_route"].sum():>18,.0f}')
print(f'  Certificate price used: EUR {CERTIFICATE_PRICE_EUR}/tCO2')

Total estimated CBAM cost (all countries, all sectors):
  High route: EUR     15,549,076,127
  Low route:  EUR     15,530,328,301
  Certificate price used: EUR 75.36/tCO2


In [11]:
# ── Build output table 1: cost by country and sector ─────────────────────────
# Grain: one row per (country, sector, cn_code).
# Most granular output table, source for all aggregations.
# Contains both high-route and low-route scenarios plus route metadata.

df_cost_by_country_sector = (
    df_merged[[
        'country', 'iso2', 'sector', 'cn_code',
        'production_route_code_high', 'production_route_high',
        'production_route_code_low',  'production_route_low',
        'has_route_variation',
        'import_tonnes', 'import_value_eur',
        'default_2026_high_route', 'embedded_co2_high_route', 'cbam_cost_eur_high_route',
        'default_2026_low_route',  'embedded_co2_low_route',  'cbam_cost_eur_low_route',
    ]]
    .copy()
    .sort_values('cbam_cost_eur_high_route', ascending=False)
    .reset_index(drop=True)
)

print(f'Output table 1 shape: {df_cost_by_country_sector.shape}')
df_cost_by_country_sector.head(10)

Output table 1 shape: (10641, 17)


,country,iso2,sector,cn_code,production_route_code_high,production_route_high,production_route_code_low,production_route_low,has_route_variation,import_tonnes,import_value_eur,default_2026_high_route,embedded_co2_high_route,cbam_cost_eur_high_route,default_2026_low_route,embedded_co2_low_route,cbam_cost_eur_low_route
0,Russia,RU,Iron and Steel,72071210,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"3,152,317.10","1,588,901,031.00",3.53,"11,130,831.67","838,819,474.61",3.53,"11,130,831.67","838,819,474.61"
1,India,IN,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,584,475.51","1,040,521,909.00",4.71,"7,459,710.71","562,163,798.79",4.71,"7,459,710.71","562,163,798.79"
2,China,CN,Iron and Steel,7308,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"959,471.44","1,922,647,772.00",6.64,"6,369,451.15","480,001,839.00",6.64,"6,369,451.15","480,001,839.00"
3,Indonesia,ID,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"641,727.83","426,219,638.00",9.05,"5,809,562.09","437,808,599.12",9.05,"5,809,562.09","437,808,599.12"
4,India,IN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,007,722.43","878,023,485.00",4.71,"4,744,357.21","357,534,758.98",4.71,"4,744,357.21","357,534,758.98"
5,Turkey,TR,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,413,742.60","875,438,013.00",2.67,"3,775,064.42","284,488,854.85",2.67,"3,775,064.42","284,488,854.85"
6,China,CN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"997,240.02","914,407,145.00",3.53,"3,515,769.69","264,948,403.61",3.53,"3,515,769.69","264,948,403.61"
7,Russia,RU,Iron and Steel,7201,NaN,NaN,NaN,NaN,False,"1,029,970.84","420,882,553.00",3.34,"3,444,222.48","259,556,605.76",3.34,"3,444,222.48","259,556,605.76"
8,South Korea,KR,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,457,477.61","1,079,314,249.00",2.33,"3,396,394.79","255,952,311.17",2.33,"3,396,394.79","255,952,311.17"
9,Vietnam,VN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF",(C),"Carbon Steel, BF-BOF",False,"1,219,943.14","958,902,345.00",2.61,"3,180,391.78","239,674,324.27",2.61,"3,180,391.78","239,674,324.27"


In [12]:
# ── Build output table 2: cost by country (all sectors aggregated) ────────────
# Grain: one row per country. All 119 CBAM countries present.
# Includes total and per-sector cost for both route scenarios,
# export value, cost as share of export value, and cost per tonne.

# Total aggregation per country
df_cost_by_country = (
    df_cost_by_country_sector
    .groupby('country', as_index=False)
    .agg(
        total_import_tonnes          = ('import_tonnes',              'sum'),
        total_import_value_eur       = ('import_value_eur',           'sum'),
        total_embedded_co2_high      = ('embedded_co2_high_route',    'sum'),
        total_embedded_co2_low       = ('embedded_co2_low_route',     'sum'),
        total_cbam_cost_high_route   = ('cbam_cost_eur_high_route',   'sum'),
        total_cbam_cost_low_route    = ('cbam_cost_eur_low_route',    'sum'),
        has_any_route_variation      = ('has_route_variation',        'any'),
    )
    .sort_values('total_cbam_cost_high_route', ascending=False)
    .reset_index(drop=True)
)

# Cost as share of total CBAM-sector export value
df_cost_by_country['cbam_cost_pct_of_export_value_high'] = (
    df_cost_by_country['total_cbam_cost_high_route']
    / df_cost_by_country['total_import_value_eur'].replace(0, float('nan'))
    * 100
).round(2)

df_cost_by_country['cbam_cost_pct_of_export_value_low'] = (
    df_cost_by_country['total_cbam_cost_low_route']
    / df_cost_by_country['total_import_value_eur'].replace(0, float('nan'))
    * 100
).round(2)

# Cost per tonne exported
df_cost_by_country['cbam_cost_per_tonne_high'] = (
    df_cost_by_country['total_cbam_cost_high_route']
    / df_cost_by_country['total_import_tonnes'].replace(0, float('nan'))
).round(2)

df_cost_by_country['cbam_cost_per_tonne_low'] = (
    df_cost_by_country['total_cbam_cost_low_route']
    / df_cost_by_country['total_import_tonnes'].replace(0, float('nan'))
).round(2)

# Per-sector cost pivots for both scenarios, merged in as columns
for scenario in ['high_route', 'low_route']:
    df_pivot = (
        df_cost_by_country_sector
        .groupby(['country', 'sector'])[f'cbam_cost_eur_{scenario}']
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )
    df_pivot.columns = [
        f'cost_{c.lower().replace(" ", "_")}_{scenario}_eur' if c != 'country' else 'country'
        for c in df_pivot.columns
    ]
    df_cost_by_country = df_cost_by_country.merge(df_pivot, on='country', how='left')

# Merge iso2 from crosswalk
df_iso2 = pd.read_sql('SELECT country, iso2 FROM country_crosswalk', con)
df_cost_by_country = df_cost_by_country.merge(df_iso2, on='country', how='left')

# Rank by total high-route cost (1 = highest exposure)
df_cost_by_country.insert(
    0, 'rank',
    df_cost_by_country['total_cbam_cost_high_route']
    .rank(ascending=False, method='min')
    .astype(int)
)

print(f'Output table 2 shape: {df_cost_by_country.shape}')
print(f'Countries with zero cost: {(df_cost_by_country["total_cbam_cost_high_route"] == 0).sum()}')
df_cost_by_country.head(10)

Output table 2 shape: (119, 24)
Countries with zero cost: 21


,rank,country,total_import_tonnes,total_import_value_eur,total_embedded_co2_high,total_embedded_co2_low,total_cbam_cost_high_route,total_cbam_cost_low_route,has_any_route_variation,cbam_cost_pct_of_export_value_high,...,cost_cement_high_route_eur,cost_fertilizers_high_route_eur,cost_hydrogen_high_route_eur,cost_iron_and_steel_high_route_eur,cost_aluminium_low_route_eur,cost_cement_low_route_eur,cost_fertilizers_low_route_eur,cost_hydrogen_low_route_eur,cost_iron_and_steel_low_route_eur,iso2
0,1,China,"8,095,975.83","15,736,927,377.00","36,573,116.70","36,571,510.94","2,756,150,074.87","2,756,029,064.24",True,17.51,...,"5,475,440.14","58,292,961.74",15.46,"2,384,230,159.19","308,151,498.35","5,354,429.50","58,292,961.74",15.46,"2,384,230,159.19",CN
1,2,Turkey,"7,434,799.28","9,835,825,490.00","23,806,323.39","23,806,323.39","1,794,044,531.03","1,794,044,531.03",False,18.24,...,"6,155,300.36","62,404,197.90",8.97,"1,554,017,893.98","171,467,129.81","6,155,300.36","62,404,197.90",8.97,"1,554,017,893.98",TR
2,3,India,"4,899,194.04","5,845,842,868.00","22,833,857.84","22,833,857.78","1,720,759,527.17","1,720,759,522.20",True,29.44,...,"59,403.07","173,495.04",0.00,"1,667,158,464.91","53,368,164.15","59,398.10","173,495.04",0.00,"1,667,158,464.91",IN
3,4,Russia,"8,951,307.94","4,612,590,851.00","22,509,783.47","22,509,783.47","1,696,337,282.09","1,696,337,282.09",False,36.78,...,0.00,"319,152,347.10",0.00,"1,316,387,810.59","60,797,124.40",0.00,"319,152,347.10",0.00,"1,316,387,810.59",RU
4,5,Ukraine,"11,546,697.32","3,287,557,505.00","12,614,285.90","12,614,285.90","950,612,585.28","950,612,585.28",False,28.92,...,"188,129,221.28","3,529,407.10",0.00,"758,014,823.71","939,133.19","188,129,221.28","3,529,407.10",0.00,"758,014,823.71",UA
5,6,Indonesia,"1,106,398.01","1,058,981,848.00","9,736,165.35","9,736,165.35","733,717,421.05","733,717,421.05",False,69.29,...,0.00,"3,540.20",0.00,"730,475,599.22","3,238,281.63",0.00,"3,540.20",0.00,"730,475,599.22",ID
6,7,South Korea,"3,634,304.40","4,426,226,645.00","9,473,206.16","9,473,206.15","713,900,816.34","713,900,815.79",True,16.13,...,348.20,"2,659,066.41",8.14,"702,510,096.24","8,731,297.34",347.65,"2,659,066.41",8.14,"702,510,096.24",KR
7,8,Vietnam,"3,413,234.68","2,780,472,268.00","9,005,206.90","9,005,206.90","678,632,391.70","678,632,391.70",False,24.41,...,0.00,"34,570.45",0.00,"675,152,494.65","3,445,326.61",0.00,"34,570.45",0.00,"675,152,494.65",VN
8,9,United Kingdom,"3,329,014.01","6,506,283,157.00","8,981,256.77","8,981,256.77","676,827,510.37","676,827,510.37",False,10.40,...,"23,249,649.15","13,313,191.15","34,723.93","578,519,548.37","61,710,397.77","23,249,649.15","13,313,191.15","34,723.93","578,519,548.37",GB
9,10,Taiwan,"2,605,715.25","3,418,418,004.00","7,872,095.20","7,872,095.20","593,241,094.43","593,241,094.43",False,17.35,...,413.67,"113,718.35",0.00,"592,632,125.51","494,836.90",413.67,"113,718.35",0.00,"592,632,125.51",TW


In [13]:
# ── Build output table 3: cost by sector (all countries aggregated) ───────────
# Grain: one row per sector.
# pct_of_total based on high-route scenario as the primary reference.

df_cost_by_sector = (
    df_cost_by_country_sector
    .groupby('sector', as_index=False)
    .agg(
        n_countries                  = ('country',                  'nunique'),
        n_cn_codes                   = ('cn_code',                  'nunique'),
        total_import_tonnes          = ('import_tonnes',            'sum'),
        total_import_value_eur       = ('import_value_eur',         'sum'),
        total_embedded_co2_high      = ('embedded_co2_high_route',  'sum'),
        total_embedded_co2_low       = ('embedded_co2_low_route',   'sum'),
        total_cbam_cost_high_route   = ('cbam_cost_eur_high_route', 'sum'),
        total_cbam_cost_low_route    = ('cbam_cost_eur_low_route',  'sum'),
    )
    .sort_values('total_cbam_cost_high_route', ascending=False)
    .reset_index(drop=True)
)

df_cost_by_sector['pct_of_total_high'] = (
    df_cost_by_sector['total_cbam_cost_high_route']
    / df_cost_by_sector['total_cbam_cost_high_route'].sum()
    * 100
).round(1)

df_cost_by_sector['pct_of_total_low'] = (
    df_cost_by_sector['total_cbam_cost_low_route']
    / df_cost_by_sector['total_cbam_cost_low_route'].sum()
    * 100
).round(1)

print(f'Output table 3 shape: {df_cost_by_sector.shape}')
print(df_cost_by_sector.to_string(index=False))

Output table 3 shape: (5, 11)
        sector  n_countries  n_cn_codes  total_import_tonnes  total_import_value_eur  total_embedded_co2_high  total_embedded_co2_low  total_cbam_cost_high_route  total_cbam_cost_low_route  pct_of_total_high  pct_of_total_low
Iron and Steel           47         200        67,716,428.14       55,806,053,839.00           165,554,602.29          165,554,602.29           12,476,194,828.29          12,476,194,828.29              80.20             80.30
     Aluminium           67          28         5,881,632.10       20,054,704,690.00            16,868,806.27           16,868,806.27            1,271,233,240.66           1,271,233,240.66               8.20              8.20
   Fertilizers           90          27        10,151,157.74        4,451,452,006.00            14,660,623.37           14,660,623.37            1,104,824,577.06           1,104,824,577.06               7.10              7.10
        Cement          100           6         6,734,810.49      

In [14]:
# ── Sanity checks before writing to database ──────────────────────────────────
# Assertions and diagnostic prints to catch obvious problems before
# committing results. Review all output carefully.

print('=== SANITY CHECKS ===')

# 1. All 119 CBAM countries present
n_countries = df_cost_by_country['country'].nunique()
assert n_countries == 119, f'Expected 119 countries, got {n_countries}'
print(f'PASS: All 119 CBAM countries present')

# 2. No negative costs in either scenario
assert (df_cost_by_country_sector['cbam_cost_eur_high_route'] >= 0).all(), \
    'Negative high-route costs found.'
assert (df_cost_by_country_sector['cbam_cost_eur_low_route'] >= 0).all(), \
    'Negative low-route costs found.'
print('PASS: No negative costs in either scenario')

# 3. High route always >= low route
assert (df_cost_by_country_sector['cbam_cost_eur_high_route']
        >= df_cost_by_country_sector['cbam_cost_eur_low_route']).all(), \
    'Low-route cost exceeds high-route cost for some rows.'
print('PASS: High route >= low route for all rows')

# 4. No nulls in key output columns
key_cols = [
    'country', 'sector', 'import_tonnes',
    'default_2026_high_route', 'default_2026_low_route',
    'cbam_cost_eur_high_route', 'cbam_cost_eur_low_route'
]
nulls = df_cost_by_country_sector[key_cols].isnull().sum()
if nulls.any():
    print(f'WARNING: Nulls found in key columns:')
    print(nulls[nulls > 0])
else:
    print('PASS: No nulls in key output columns')

# 5. All five expected sectors present
expected_sectors = {'Aluminium', 'Cement', 'Fertilizers', 'Hydrogen', 'Iron and Steel'}
missing = expected_sectors - set(df_cost_by_country_sector['sector'].unique())
if missing:
    print(f'WARNING: Expected sectors missing: {missing}')
else:
    print('PASS: All five sectors present')

# 6. Countries with zero cost
zero_cost = df_cost_by_country[
    df_cost_by_country['total_cbam_cost_high_route'] == 0
]['country'].tolist()
print(f'\nCountries with zero CBAM cost: {len(zero_cost)}')
print(zero_cost)

# 7. Route variation summary
n_varied = df_cost_by_country_sector['has_route_variation'].sum()
print(f'\nRows with route variation: {n_varied:,} of {len(df_cost_by_country_sector):,}')
print(f'Countries with any route variation: {df_cost_by_country["has_any_route_variation"].sum()}')

# 8. Top 10 countries by high-route cost
print('\nTop 10 countries by estimated CBAM cost (high route):')
print(
    df_cost_by_country[[
        'rank', 'country',
        'total_cbam_cost_high_route', 'total_cbam_cost_low_route'
    ]]
    .head(10)
    .to_string(index=False)
)

=== SANITY CHECKS ===
PASS: All 119 CBAM countries present
PASS: No negative costs in either scenario
PASS: High route >= low route for all rows
PASS: No nulls in key output columns
PASS: All five sectors present

Countries with zero CBAM cost: 21
['Curacao', 'Brunei Darussalam', 'Congo', 'Cambodia', 'Eritrea', 'Yemen', 'Equatorial Guinea', 'Rwanda', 'Eswatini', 'Haiti', 'Papua New Guinea', 'Jamaica', 'Laos', 'Suriname', 'Sudan', 'Mongolia', 'Namibia', 'Sierra Leone', 'Nepal', 'North Korea', 'Mali']

Rows with route variation: 28 of 10,641
Countries with any route variation: 17

Top 10 countries by estimated CBAM cost (high route):
 rank        country  total_cbam_cost_high_route  total_cbam_cost_low_route
    1          China            2,756,150,074.87           2,756,029,064.24
    2         Turkey            1,794,044,531.03           1,794,044,531.03
    3          India            1,720,759,527.17           1,720,759,522.20
    4         Russia            1,696,337,282.09        

In [15]:
# ── Write output tables to database ──────────────────────────────────────────
# if_exists='replace' makes this notebook fully idempotent and safe to re-run.

tables = {
    'cbam_cost_by_country_sector': df_cost_by_country_sector,
    'cbam_cost_by_country':        df_cost_by_country,
    'cbam_cost_by_sector':         df_cost_by_sector,
}

for table_name, df in tables.items():
    df.to_sql(table_name, con, if_exists='replace', index=False)
    print(f'Written: {table_name} ({len(df):,} rows)')

print('\nAll output tables written successfully.')

Written: cbam_cost_by_country_sector (10,641 rows)
Written: cbam_cost_by_country (119 rows)
Written: cbam_cost_by_sector (5 rows)

All output tables written successfully.


In [16]:
# ── Verification queries ──────────────────────────────────────────────────────
# Confirm all three tables exist in the database with correct row counts.

print('=== DATABASE VERIFICATION ===')
for table_name, df in tables.items():
    count    = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table_name}', con).iloc[0, 0]
    expected = len(df)
    status   = 'PASS' if count == expected else 'FAIL'
    print(f'[{status}] {table_name}: {count:,} rows (expected {expected:,})')

con.close()
print('\nConnection closed. Notebook complete.')

=== DATABASE VERIFICATION ===
[PASS] cbam_cost_by_country_sector: 10,641 rows (expected 10,641)
[PASS] cbam_cost_by_country: 119 rows (expected 119)
[PASS] cbam_cost_by_sector: 5 rows (expected 5)

Connection closed. Notebook complete.
